In [1]:
import numpy as np
import pandas as pd
import OpenEXR
import re
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

In [2]:
df1 = pd.read_csv(r"C:\Users\sol77\Desktop\ML-For-CV-Robustness\datasets\datasetGTNoTrees.csv")
df2 = pd.read_csv(r"C:\Users\sol77\Desktop\datasetGTTreesIllumShift.csv")

In [3]:
df1

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
333475,0.262388,0.000241,0.000365,550.956812,480.270977,15.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
333476,0.239909,0.000271,0.000399,550.956812,480.270977,16.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
333477,0.284332,0.000270,0.000425,550.956812,480.270977,17.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
333478,0.241989,0.000346,0.000511,550.956812,480.270977,18.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26


In [4]:
df2

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.224674,0.000277,0.000397,1141.969,1031.228,0.0,180.0,-63.812729,-56.074120,0.0,-19.350,-0.0,0.20
1,0.196116,0.000391,0.000550,1141.969,1031.228,1.0,180.0,-63.812729,-56.074120,0.0,-19.350,-0.0,0.20
2,0.202034,0.000388,0.000555,1141.969,1031.228,2.0,180.0,-63.812729,-56.074120,0.0,-19.350,-0.0,0.20
3,0.185897,0.000346,0.000477,1141.969,1031.228,3.0,180.0,-63.812729,-56.074120,0.0,-19.350,-0.0,0.20
4,0.233005,0.000312,0.000456,1141.969,1031.228,4.0,180.0,-63.812729,-56.074120,0.0,-19.350,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
332635,0.261726,0.000240,0.000362,1.226,1.498,15.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.34
332636,0.239247,0.000270,0.000396,1.226,1.498,16.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.34
332637,0.283672,0.000269,0.000422,1.226,1.498,17.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.34
332638,0.241322,0.000345,0.000508,1.226,1.498,18.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.34


In [5]:
big_df = pd.concat([df1, df2], ignore_index=True)
big_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
666115,0.261726,0.000240,0.000362,1.226000,1.498000,15.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.34
666116,0.239247,0.000270,0.000396,1.226000,1.498000,16.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.34
666117,0.283672,0.000269,0.000422,1.226000,1.498000,17.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.34
666118,0.241322,0.000345,0.000508,1.226000,1.498000,18.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.34


In [6]:
big_df.to_csv(r"C:\Users\sol77\Desktop\datasetGTCombinedIlumShift.csv", index=False)